# Greek Reporting Obligations — train / select (Colab)

**Runtime → Change runtime type → T4 GPU.** Run the cells top to bottom.

One round = train (cell 4) → select (cell 5) → annotate the xlsx → next round with `ROUND += 1`.

- Data and results are on Google Drive; model checkpoints go to `/content/models` (local, deleted when the session ends).
- So run **train and select in the same session**. If the session dropped, re-run train first (a few minutes).
- Never edit `data/2026_10_22/` (frozen gold set).

In [ ]:
# 1. mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. paths — set BASE to where the project folder is in YOUR Drive
#    (if the folder was shared with you: Drive -> right-click -> Organize -> Add shortcut to My Drive)
import os
BASE = "/content/drive/MyDrive/EMP/ΔΙΠΛΩΜΑΤΙΚΕΣ/23. vroutsis- GreekLegalReporting"
assert os.path.isdir(BASE), f"not found: {BASE} — look under /content/drive/MyDrive"
os.environ["GRRO_BASE"] = BASE
os.environ["GRRO_MODELS"] = "/content/models"
os.chdir(BASE)
!ls data/2026_10_22 data/pool data/rounds
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. dependencies (torch / transformers / pandas are preinstalled on Colab)
!pip -q install datasets accelerate openpyxl

In [ ]:
# 4. train — ROUND 0 = gold train only; ROUND N also uses data/rounds/round_01..N_labelled.xlsx
ROUND = 0
QUICK_CHECK = False   # True: one seed, no checkpoints (just to see it runs)

if QUICK_CHECK:
    !python code/train_bert_gr.py --round {ROUND} --seeds 42 --no_save
else:
    !python code/train_bert_gr.py --round {ROUND}

In [ ]:
# 4b. results of this round (test = frozen 60 gold rows)
import json
m = json.load(open(f"results/round_{ROUND:02d}/metrics.json", encoding="utf-8"))
t, b = m["test_mean_sd"], m["test_rule_baseline"]
print(f"round {ROUND} | train {m['n_train']} (RO {m['n_train_ro']})")
for k in ("f1", "precision", "recall", "kappa"):
    print(f"  {k:9s} model {t[k]['mean']:.3f} ± {t[k]['sd']:.3f}   rules {b[k]:.3f}")

In [ ]:
# 5. select next batch — needs data/pool/ro_candidates_pool_v1.json and the checkpoints from cell 4
!python code/select_round.py --round {ROUND}

## Annotation (after cell 5)

1. Open `data/rounds/round_{ROUND+1:02d}_to_label.xlsx`.
2. For every row fill `Evaluation` = `RO` or `NOT_RO` (pre-filled rows: confirm or change; empty rows: decide yourself — do not skip them) and `Reason` (same style as sheet Τελικό).
3. Follow `data/2026_08_26/annotation_rubrik_new.md`; use `previous_sentence` / `next_sentence` when the sentence is cut.
4. Save as `data/rounds/round_{ROUND+1:02d}_labelled.xlsx` (same columns, do not rename them).
5. Back in this notebook: set `ROUND = ROUND + 1` in cell 4, run cells 1–5 again.

Stop when test F1 (cell 4b) no longer improves between rounds.